In [ ]:
# ربط Drive والتحقق من GPU والبيانات وصيغة ملفات التوقيت
import os, glob, torch
from google.colab import drive
drive.mount('/content/drive')

print("CUDA:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")

ROOT = '/content/drive/MyDrive'
def find_dir(name, depth=4):
    for d in range(depth):
        hits = glob.glob(os.path.join(ROOT, *(['*'] * d), name))
        hits = [h for h in hits if os.path.isdir(h)]
        if hits: return hits[0]
    return None

TRAIN_DIR = find_dir('train.7z')
TEST_DIR  = find_dir('test.7z')
SPEC_DIR  = find_dir('specs_v2')
TRAIN_DIR = os.path.join(TRAIN_DIR, 'train') if TRAIN_DIR and os.path.isdir(os.path.join(TRAIN_DIR, 'train')) else TRAIN_DIR
TEST_DIR  = os.path.join(TEST_DIR, 'test') if TEST_DIR and os.path.isdir(os.path.join(TEST_DIR, 'test')) else TEST_DIR
print("TRAIN_DIR:", TRAIN_DIR, "\nTEST_DIR :", TEST_DIR, "\nSPEC_DIR :", SPEC_DIR)

for name, d in [('Train', TRAIN_DIR), ('Test', TEST_DIR)]:
    print(f"{name}: wav={len(glob.glob(os.path.join(d, '*.wav')))} | label={len(glob.glob(os.path.join(d, '*_label.txt')))}")

if SPEC_DIR:
    print("specs_v2 contents:", sorted(os.listdir(SPEC_DIR))[:15])

sample = sorted(glob.glob(os.path.join(TRAIN_DIR, '*_label.txt')))[0]
print("\nSample:", os.path.basename(sample))
with open(sample) as f:
    for line in f.readlines()[:8]:
        print(repr(line))

Mounted at /content/drive
CUDA: True | NVIDIA L4
TRAIN_DIR: /content/drive/MyDrive/train.7z/train 
TEST_DIR : /content/drive/MyDrive/test.7z/test 
SPEC_DIR : /content/drive/MyDrive/specs_v2
Train: wav=7809 | label=7809
Test: wav=1956 | label=1956
specs_v2 contents: ['ablation_table.csv', 'best_model_v2.pt', 'best_model_v3.pt', 'best_model_v4.pt', 'fold0.pt', 'fold0_cas.pt', 'fold1.pt', 'fold1_cas.pt', 'fold2.pt', 'fold2_cas.pt', 'fold3.pt', 'fold3_cas.pt', 'fold4.pt', 'fold4_cas.pt', 'gradcam_6classes.png']

Sample: steth_20180814_09_37_11_label.txt
'I 00:00:01.500 00:00:02.457\n'
'D 00:00:01.500 00:00:02.457\n'
'E 00:00:02.608 00:00:03.637\n'
'D 00:00:02.608 00:00:03.637\n'
'I 00:00:05.227 00:00:06.350\n'
'D 00:00:05.227 00:00:06.350\n'
'E 00:00:06.431 00:00:07.411\n'
'D 00:00:06.431 00:00:07.411\n'


In [ ]:
# تحويل الصوت لـ Mel-Spectrogram وبناء labels زمنية لكل إطار وحفظها مرة واحدة
import os, glob, shutil, numpy as np, librosa
from joblib import Parallel, delayed

SR, N_FFT, HOP, N_MELS, FMIN, FMAX, T = 4000, 512, 64, 128, 50, 2000, 938
MEAN, STD = -8.5267, 2.4603
CLASSES = ['I', 'E', 'D', 'Wheeze', 'Stridor', 'Rhonchi']
C2I = {c: i for i, c in enumerate(CLASSES)}
FRAME_SEC = HOP / SR  # 0.016 s

LOCAL = '/content/cache'; DRIVE_CACHE = '/content/drive/MyDrive/crnn_cache'
os.makedirs(LOCAL, exist_ok=True); os.makedirs(DRIVE_CACHE, exist_ok=True)

def to_sec(t):
    h, m, s = t.split(':'); return int(h) * 3600 + int(m) * 60 + float(s)

def process(wav_path):
    y, _ = librosa.load(wav_path, sr=SR, mono=True)
    y = np.pad(y, (0, max(0, SR * 15 - len(y))))[:SR * 15]
    mel = librosa.feature.melspectrogram(y=y, sr=SR, n_fft=N_FFT, hop_length=HOP,
                                         n_mels=N_MELS, fmin=FMIN, fmax=FMAX)
    x = ((np.log(mel + 1e-5) - MEAN) / STD)[:, :T]
    x = np.pad(x, ((0, 0), (0, T - x.shape[1]))).astype(np.float16)
    yf = np.zeros((T, 6), np.uint8); unk = []
    with open(wav_path.replace('.wav', '_label.txt')) as f:
        for line in f:
            p = line.split()
            if len(p) != 3: continue
            if p[0] not in C2I: unk.append(p[0]); continue
            s = int(np.floor(to_sec(p[1]) / FRAME_SEC)); e = int(np.ceil(to_sec(p[2]) / FRAME_SEC))
            yf[max(0, s):min(T, e), C2I[p[0]]] = 1
    return x, yf, unk

for split, d in [('train', TRAIN_DIR), ('test', TEST_DIR)]:
    if os.path.exists(f'{DRIVE_CACHE}/{split}_X.npy'):
        for k in ['X', 'Yf', 'names']:
            shutil.copy(f'{DRIVE_CACHE}/{split}_{k}.npy', LOCAL)
        print(f'{split}: loaded from Drive cache'); continue
    wavs = sorted(glob.glob(os.path.join(d, '*.wav')))
    out = Parallel(n_jobs=os.cpu_count(), verbose=5)(delayed(process)(w) for w in wavs)
    X = np.stack([o[0] for o in out]); Yf = np.stack([o[1] for o in out])
    names = np.array([os.path.basename(w)[:-4] for w in wavs])
    unk = sorted(set(u for o in out for u in o[2]))
    for k, v in [('X', X), ('Yf', Yf), ('names', names)]:
        np.save(f'{LOCAL}/{split}_{k}.npy', v); shutil.copy(f'{LOCAL}/{split}_{k}.npy', DRIVE_CACHE)
    clip = Yf.max(1).sum(0); frame = Yf.mean((0, 1))
    print(f'\n{split}: X={X.shape} Yf={Yf.shape} | unknown tags={unk}')
    print('  clip-level positives :', dict(zip(CLASSES, clip.tolist())))
    print('  frame-level positive %:', dict(zip(CLASSES, np.round(frame * 100, 2).tolist())))

[Parallel(n_jobs=12)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  48 tasks      | elapsed:   34.5s


In [ ]:
# Idea: matched retraining of CNN-BiGRU (from scratch) under the BEATs protocol
# Goal: same 382-session folds, 7 outputs (6 classes + Abnormal), <=25 epochs, patience 5, selection S
# Source: architecture after Hsu et al., PLOS ONE 2021
from google.colab import drive; drive.mount('/content/drive')
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F, os, time, math
C   = '/content/drive/MyDrive/crnn_cache'
OUT = '/content/drive/MyDrive/matched_baselines/crnn'; os.makedirs(OUT, exist_ok=True)
dev = 'cuda'

# ---- data (cache is already z-scored log-mel, 4 kHz, 128 x 938)
Xtr = np.load(f'{C}/train_X.npy'); Xte = np.load(f'{C}/test_X.npy')
def add_abn(Yf):                                   # frame labels (N,938,6) -> (N,938,7)
    Yf = np.asarray(Yf, dtype=np.uint8); assert Yf.shape[1:] == (938, 6), Yf.shape
    return np.concatenate([Yf, Yf[..., 2:6].max(-1, keepdims=True)], -1)
Yf_tr = add_abn(np.load(f'{C}/train_Yf.npy')); Yf_te = add_abn(np.load(f'{C}/test_Yf.npy'))
Ytr, Yte = Yf_tr.max(1), Yf_te.max(1)              # recording-level labels
fold = np.load(f'{C}/train_fold.npy')
print('train', Xtr.shape, '| test', Xte.shape, '| positives', Ytr.sum(0), '| fold sizes', np.bincount(fold))

# ---- model: 4 conv blocks (frequency-only pooling) -> Linear 256 -> 2-layer BiGRU -> 7 frame logits
def block(ci, co):
    return nn.Sequential(nn.Conv2d(ci, co, 3, padding=1, bias=False), nn.BatchNorm2d(co), nn.ReLU(inplace=True),
                         nn.Conv2d(co, co, 3, padding=1, bias=False), nn.BatchNorm2d(co), nn.ReLU(inplace=True),
                         nn.MaxPool2d((2, 1)))
class CRNN(nn.Module):
    def __init__(s, n=7):
        super().__init__()
        s.cnn  = nn.Sequential(block(1, 32), block(32, 64), block(64, 128), block(128, 128))   # 128 mel -> 8
        s.proj = nn.Sequential(nn.Linear(128 * 8, 256), nn.ReLU(inplace=True))
        s.gru  = nn.GRU(256, 128, num_layers=2, batch_first=True, bidirectional=True, dropout=0.3)
        s.head = nn.Linear(256, n)
    def forward(s, x):                                            # x: (B,128,938)
        h = s.cnn(x[:, None])                                     # (B,128,8,938)
        h = h.permute(0, 3, 1, 2).flatten(2)                      # (B,938,1024) channel x frequency
        h, _ = s.gru(s.proj(h))
        return s.head(h)                                          # frame logits (B,938,7)

m = CRNN().to(dev)
print('params: %.2f M' % (sum(p.numel() for p in m.parameters()) / 1e6))
with torch.no_grad():
    out = m(torch.from_numpy(Xtr[:4].astype(np.float32)).to(dev))
print('output', tuple(out.shape)); del m, out

Mounted at /content/drive
train (7809, 128, 938) | test (1956, 128, 938) | positives [7510 4593 3076 2253  253  944 4999] | fold sizes [2051 1334 1341 1510 1573]
params: 1.44 M
output (4, 938, 7)


In [ ]:
# Idea: CNN-BiGRU training under the matched protocol (5 folds, resumable after disconnect)
# Goal: per-fold best model by S = 0.5*(6-class Macro-AUC + Abnormal AUC) on validation
# Source: frame+recording BCE (Hsu et al. 2021); SpecAugment (Park et al. 2019); AdamW (Loshchilov & Hutter 2019)
from sklearn.metrics import roc_auc_score
EPOCHS, PAT, BS, WARM = 25, 5, 32, 2

def rec_prob(z):                                   # frame logits (B,938,7) -> recording prob (B,7)
    p = torch.sigmoid(z.float()).transpose(1, 2)
    return F.avg_pool1d(p, 9, 1, 4).amax(-1)        # 9-frame moving average, max over time

def specaug(x):                                    # 2 frequency masks (<=16), 1 time mask (<=40), per sample
    B, Fq, T = x.shape; d = x.device
    for size, n, axis in [(16, 2, 1), (40, 1, 2)]:
        L = Fq if axis == 1 else T
        r = torch.arange(L, device=d).view((1, L, 1) if axis == 1 else (1, 1, L))
        for _ in range(n):
            w  = torch.randint(0, size + 1, (B, 1, 1), device=d)
            s0 = (torch.rand(B, 1, 1, device=d) * (L - w)).long()
            x  = x.masked_fill((r >= s0) & (r < s0 + w), 0.)
    return x

def score(y, p):
    auc = [roc_auc_score(y[:, c], p[:, c]) for c in range(7)]
    return 0.5 * (np.mean(auc[:6]) + auc[6]), np.mean(auc[:6])

def predict(model, X, bs=64):
    model.eval(); out = []
    with torch.no_grad(), torch.autocast('cuda', dtype=torch.float16):
        for b in range(0, len(X), bs):
            out.append(rec_prob(model(torch.from_numpy(X[b:b+bs].astype(np.float32)).to(dev))).cpu().numpy())
    return np.concatenate(out)

for k in range(5):
    if os.path.exists(f'{OUT}/fold{k}_val_prob.npy'): print(f'fold {k} already done'); continue
    torch.manual_seed(42 + k)
    tr, va = np.where(fold != k)[0], np.where(fold == k)[0]
    pf = Yf_tr[tr].mean((0, 1)); pc = Ytr[tr].mean(0)
    pw_f = torch.tensor(np.clip((1 - pf) / np.maximum(pf, 1e-6), 1, 10), dtype=torch.float32, device=dev)
    pw_c = torch.tensor(np.clip((1 - pc) / np.maximum(pc, 1e-6), 1, 10), dtype=torch.float32, device=dev)

    m = CRNN().to(dev)
    opt = torch.optim.AdamW(m.parameters(), lr=1e-3, weight_decay=1e-4)
    spe = len(tr) // BS; tot = EPOCHS * spe; wu = WARM * spe
    sch = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: s / wu if s < wu else 0.5 * (1 + math.cos(math.pi * (s - wu) / (tot - wu))))
    scaler = torch.amp.GradScaler('cuda')

    rf = f'{OUT}/fold{k}_resume.pt'; ep0, best, bad, vbest = 0, -1., 0, None
    if os.path.exists(rf):
        ck = torch.load(rf, map_location='cpu', weights_only=False)
        m.load_state_dict(ck['m']); opt.load_state_dict(ck['o']); sch.load_state_dict(ck['s']); scaler.load_state_dict(ck['sc'])
        ep0, best, bad, vbest = ck['ep'] + 1, ck['best'], ck['bad'], ck['vp']; print(f'fold {k} resumed at epoch {ep0}')

    for ep in range(ep0, EPOCHS):
        t0 = time.time(); m.train(); rng = np.random.default_rng(1000 * k + ep); perm = rng.permutation(tr)
        for i in range(spe):
            idx = np.sort(perm[i*BS:(i+1)*BS])
            xb = specaug(torch.from_numpy(Xtr[idx].astype(np.float32)).to(dev))
            yf = torch.from_numpy(Yf_tr[idx]).float().to(dev); yc = torch.from_numpy(Ytr[idx]).float().to(dev)
            with torch.autocast('cuda', dtype=torch.float16):
                z = m(xb)
            z = z.float(); p = rec_prob(z).clamp(1e-6, 1 - 1e-6)
            loss = F.binary_cross_entropy_with_logits(z, yf, pos_weight=pw_f) \
                   - (pw_c * yc * torch.log(p) + (1 - yc) * torch.log(1 - p)).mean()
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
            scaler.unscale_(opt); nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sch.step()
        vp = predict(m, Xtr[va]); S, a6 = score(Ytr[va], vp)
        if S > best:
            best, bad, vbest = S, 0, vp
            torch.save(m.state_dict(), f'{OUT}/fold{k}_best.pt')
        else:
            bad += 1
        torch.save({'m': m.state_dict(), 'o': opt.state_dict(), 's': sch.state_dict(), 'sc': scaler.state_dict(),
                    'ep': ep, 'best': best, 'bad': bad, 'vp': vbest}, rf)
        print(f'fold {k} ep {ep:2d} | loss {loss.item():.3f} | S {S:.4f} (6-class AUC {a6:.4f}) | best {best:.4f} | {time.time()-t0:.0f}s')
        if bad >= PAT: break
    np.save(f'{OUT}/fold{k}_val_prob.npy', vbest); os.remove(rf)
    print(f'== fold {k} finished: best S {best:.4f}')
    del m, opt; torch.cuda.empty_cache()

fold 0 ep  0 | loss 1.475 | S 0.6733 (6-class AUC 0.6922) | best 0.6733 | 109s
fold 0 ep  1 | loss 1.129 | S 0.7244 (6-class AUC 0.7221) | best 0.7244 | 110s
fold 0 ep  2 | loss 1.039 | S 0.7699 (6-class AUC 0.7479) | best 0.7699 | 110s
fold 0 ep  3 | loss 1.146 | S 0.8084 (6-class AUC 0.7870) | best 0.8084 | 110s
fold 0 ep  4 | loss 1.252 | S 0.8340 (6-class AUC 0.8196) | best 0.8340 | 110s
fold 0 ep  5 | loss 0.789 | S 0.8470 (6-class AUC 0.8367) | best 0.8470 | 110s
fold 0 ep  6 | loss 1.023 | S 0.8286 (6-class AUC 0.8193) | best 0.8470 | 110s
fold 0 ep  7 | loss 0.916 | S 0.8434 (6-class AUC 0.8342) | best 0.8470 | 110s
fold 0 ep  8 | loss 0.877 | S 0.8455 (6-class AUC 0.8383) | best 0.8470 | 110s
fold 0 ep  9 | loss 0.815 | S 0.8518 (6-class AUC 0.8365) | best 0.8518 | 110s
fold 0 ep 10 | loss 1.038 | S 0.8520 (6-class AUC 0.8466) | best 0.8520 | 110s
fold 0 ep 11 | loss 0.685 | S 0.8690 (6-class AUC 0.8586) | best 0.8690 | 110s
fold 0 ep 12 | loss 0.738 | S 0.8616 (6-class AUC 0.

In [ ]:
# Idea: matched evaluation of CNN-BiGRU under the BEATs protocol
# Goal: 5-fold ensemble on the 1,956 test recordings + OOF Accuracy/F1-optimal thresholds
# Source: threshold moving on OOF (Lipton et al. 2014)
import pandas as pd
CLS = ['Inhalation','Exhalation','Crackle','Wheeze','Stridor','Rhonchi','Abnormal']

# 1) OOF probabilities from saved validation predictions
oof = np.zeros(Ytr.shape, dtype=np.float32)
for k in range(5):
    oof[fold == k] = np.load(f'{OUT}/fold{k}_val_prob.npy')

# 2) Test probabilities = mean of the 5 fold models
probs = []
for k in range(5):
    m = CRNN().to(dev); m.load_state_dict(torch.load(f'{OUT}/fold{k}_best.pt', map_location=dev))
    probs.append(predict(m, Xte)); print('fold', k, 'predicted'); del m
P = np.mean(probs, 0); np.save(f'{OUT}/test_prob_5fold.npy', P)

# 3) Per-class thresholds on OOF (grid 0.01-0.99)
grid = np.round(np.arange(0.01, 1.0, 0.01), 2)
def best_t(y, p, obj):
    sc = []
    for t in grid:
        yp = p >= t; tp = (yp & (y == 1)).sum(); fp = (yp & (y == 0)).sum(); fn = (~yp & (y == 1)).sum()
        sc.append((yp == y).mean() if obj == 'acc' else 2*tp / max(2*tp + fp + fn, 1))
    return grid[int(np.argmax(sc))]
T = {o: np.array([best_t(Ytr[:, c], oof[:, c], o) for c in range(7)]) for o in ['acc', 'f1']}

# 4) Test metrics
pd.set_option('display.precision', 3)
allrows = []
for o in ['acc', 'f1']:
    rows = []
    for c in range(7):
        y, p = Yte[:, c], P[:, c]; yp = p >= T[o][c]
        tp = (yp & (y == 1)).sum(); fp = (yp & (y == 0)).sum(); fn = (~yp & (y == 1)).sum()
        pr = tp / max(tp + fp, 1); rc = tp / max(tp + fn, 1)
        rows.append(dict(Class=CLS[c], Threshold=T[o][c], Accuracy=(yp == y).mean(), Precision=pr,
                         Recall=rc, F1=2*pr*rc / max(pr + rc, 1e-9), AUC=roc_auc_score(y, p)))
    df = pd.DataFrame(rows); df.insert(0, 'Objective', o); allrows.append(df)
    print(f'\n=== {o.upper()}-optimal thresholds ==='); print(df.drop(columns='Objective').to_string(index=False))
    mac = df[df.Class != 'Abnormal'][['Accuracy','Precision','Recall','F1','AUC']].mean().add_prefix('Macro-')
    print(mac.to_frame().T.to_string(index=False))
pd.concat(allrows).to_csv(f'{OUT}/crnn_test_metrics.csv', index=False)
print('\nOOF 6-class Macro-AUC:', round(np.mean([roc_auc_score(Ytr[:, c], oof[:, c]) for c in range(6)]), 4))

fold 0 predicted
fold 1 predicted
fold 2 predicted
fold 3 predicted
fold 4 predicted

=== ACC-optimal thresholds ===
     Class  Threshold  Accuracy  Precision  Recall    F1   AUC
Inhalation       0.11     0.989      0.989   1.000 0.995 0.857
Exhalation       0.60     0.718      0.691   0.759 0.723 0.800
   Crackle       0.68     0.748      0.375   0.505 0.431 0.721
    Wheeze       0.68     0.826      0.655   0.333 0.442 0.821
   Stridor       0.96     0.994      0.000   0.000 0.000 0.809
   Rhonchi       0.92     0.860      0.767   0.111 0.195 0.592
  Abnormal       0.58     0.709      0.722   0.658 0.689 0.778
 Macro-Accuracy  Macro-Precision  Macro-Recall  Macro-F1  Macro-AUC
          0.856             0.58         0.452     0.464      0.767

=== F1-optimal thresholds ===
     Class  Threshold  Accuracy  Precision  Recall    F1   AUC
Inhalation       0.11     0.989      0.989   1.000 0.995 0.857
Exhalation       0.50     0.678      0.624   0.846 0.718 0.800
   Crackle       0.57  

In [ ]:
# Idea: matched InceptionTime baseline (from scratch) on the same 15-s log-mel input
# Goal: same input, folds, 7 outputs, loss, budget and selection as BEATs/AST/EfficientAT/CNN-BiGRU
# Source: Ismail Fawaz et al., InceptionTime, Data Min. Knowl. Discov. 2020
OUT = '/content/drive/MyDrive/matched_baselines/inceptiontime'; os.makedirs(OUT, exist_ok=True)

class Inception(nn.Module):                         # bottleneck + 3 parallel kernels + max-pool branch
    def __init__(s, ci, nf=32, ks=(39, 19, 9), bott=32):
        super().__init__()
        s.bott  = nn.Conv1d(ci, bott, 1, bias=False)
        s.convs = nn.ModuleList([nn.Conv1d(bott, nf, k, padding=k // 2, bias=False) for k in ks])
        s.mp    = nn.Sequential(nn.MaxPool1d(3, 1, 1), nn.Conv1d(ci, nf, 1, bias=False))
        s.bn    = nn.BatchNorm1d(nf * 4)
    def forward(s, x):
        b = s.bott(x)
        return F.relu(s.bn(torch.cat([c(b) for c in s.convs] + [s.mp(x)], 1)))

class InceptionTime(nn.Module):                     # 6 modules, residual every 3, frame-level head
    def __init__(s, ci=128, n=7, depth=6, nf=32):
        super().__init__()
        s.mods = nn.ModuleList([Inception(ci if d == 0 else nf * 4, nf) for d in range(depth)])
        s.sc   = nn.ModuleList([nn.Sequential(nn.Conv1d(ci if j == 0 else nf * 4, nf * 4, 1, bias=False),
                                              nn.BatchNorm1d(nf * 4)) for j in range(depth // 3)])
        s.head = nn.Linear(nf * 4, n)
    def forward(s, x):                                            # x: (B,128,938), mel bands as channels
        res = h = x
        for d, m in enumerate(s.mods):
            h = m(h)
            if d % 3 == 2: h = F.relu(h + s.sc[d // 3](res)); res = h
        return s.head(h.transpose(1, 2))                          # frame logits (B,938,7)

m = InceptionTime().to(dev)
print('params: %.2f M' % (sum(p.numel() for p in m.parameters()) / 1e6))
with torch.no_grad():
    out = m(torch.from_numpy(Xtr[:4].astype(np.float32)).to(dev))
print('output', tuple(out.shape)); del m, out

params: 0.50 M
output (4, 938, 7)


In [ ]:
# Idea: InceptionTime training under the matched protocol (5 folds, resumable after disconnect)
# Goal: per-fold best model by S = 0.5*(6-class Macro-AUC + Abnormal AUC) on validation
# Source: Ismail Fawaz et al. 2020; frame+recording BCE (Hsu et al. 2021); SpecAugment (Park et al. 2019)
for k in range(5):
    if os.path.exists(f'{OUT}/fold{k}_val_prob.npy'): print(f'fold {k} already done'); continue
    torch.manual_seed(42 + k)
    tr, va = np.where(fold != k)[0], np.where(fold == k)[0]
    pf = Yf_tr[tr].mean((0, 1)); pc = Ytr[tr].mean(0)
    pw_f = torch.tensor(np.clip((1 - pf) / np.maximum(pf, 1e-6), 1, 10), dtype=torch.float32, device=dev)
    pw_c = torch.tensor(np.clip((1 - pc) / np.maximum(pc, 1e-6), 1, 10), dtype=torch.float32, device=dev)

    m = InceptionTime().to(dev)
    opt = torch.optim.AdamW(m.parameters(), lr=1e-3, weight_decay=1e-4)
    spe = len(tr) // BS; tot = EPOCHS * spe; wu = WARM * spe
    sch = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: s / wu if s < wu else 0.5 * (1 + math.cos(math.pi * (s - wu) / (tot - wu))))
    scaler = torch.amp.GradScaler('cuda')

    rf = f'{OUT}/fold{k}_resume.pt'; ep0, best, bad, vbest = 0, -1., 0, None
    if os.path.exists(rf):
        ck = torch.load(rf, map_location='cpu', weights_only=False)
        m.load_state_dict(ck['m']); opt.load_state_dict(ck['o']); sch.load_state_dict(ck['s']); scaler.load_state_dict(ck['sc'])
        ep0, best, bad, vbest = ck['ep'] + 1, ck['best'], ck['bad'], ck['vp']; print(f'fold {k} resumed at epoch {ep0}')

    for ep in range(ep0, EPOCHS):
        t0 = time.time(); m.train(); rng = np.random.default_rng(1000 * k + ep); perm = rng.permutation(tr)
        for i in range(spe):
            idx = np.sort(perm[i*BS:(i+1)*BS])
            xb = specaug(torch.from_numpy(Xtr[idx].astype(np.float32)).to(dev))
            yf = torch.from_numpy(Yf_tr[idx]).float().to(dev); yc = torch.from_numpy(Ytr[idx]).float().to(dev)
            with torch.autocast('cuda', dtype=torch.float16):
                z = m(xb)
            z = z.float(); p = rec_prob(z).clamp(1e-6, 1 - 1e-6)
            loss = F.binary_cross_entropy_with_logits(z, yf, pos_weight=pw_f) \
                   - (pw_c * yc * torch.log(p) + (1 - yc) * torch.log(1 - p)).mean()
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
            scaler.unscale_(opt); nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sch.step()
        vp = predict(m, Xtr[va]); S, a6 = score(Ytr[va], vp)
        if S > best:
            best, bad, vbest = S, 0, vp
            torch.save(m.state_dict(), f'{OUT}/fold{k}_best.pt')
        else:
            bad += 1
        torch.save({'m': m.state_dict(), 'o': opt.state_dict(), 's': sch.state_dict(), 'sc': scaler.state_dict(),
                    'ep': ep, 'best': best, 'bad': bad, 'vp': vbest}, rf)
        print(f'fold {k} ep {ep:2d} | loss {loss.item():.3f} | S {S:.4f} (6-class AUC {a6:.4f}) | best {best:.4f} | {time.time()-t0:.0f}s')
        if bad >= PAT: break
    np.save(f'{OUT}/fold{k}_val_prob.npy', vbest); os.remove(rf)
    print(f'== fold {k} finished: best S {best:.4f}')
    del m, opt; torch.cuda.empty_cache()

fold 0 ep  0 | loss 1.521 | S 0.7082 (6-class AUC 0.7026) | best 0.7082 | 17s
fold 0 ep  1 | loss 1.372 | S 0.7029 (6-class AUC 0.6890) | best 0.7082 | 9s
fold 0 ep  2 | loss 1.148 | S 0.7412 (6-class AUC 0.7212) | best 0.7412 | 10s
fold 0 ep  3 | loss 1.332 | S 0.7120 (6-class AUC 0.7198) | best 0.7412 | 10s
fold 0 ep  4 | loss 1.489 | S 0.7393 (6-class AUC 0.7482) | best 0.7412 | 9s
fold 0 ep  5 | loss 1.151 | S 0.7658 (6-class AUC 0.7638) | best 0.7658 | 11s
fold 0 ep  6 | loss 1.231 | S 0.7703 (6-class AUC 0.7615) | best 0.7703 | 10s
fold 0 ep  7 | loss 1.306 | S 0.7589 (6-class AUC 0.7487) | best 0.7703 | 10s
fold 0 ep  8 | loss 1.175 | S 0.7846 (6-class AUC 0.7853) | best 0.7846 | 10s
fold 0 ep  9 | loss 1.208 | S 0.7772 (6-class AUC 0.7870) | best 0.7846 | 10s
fold 0 ep 10 | loss 1.359 | S 0.7932 (6-class AUC 0.7927) | best 0.7932 | 10s
fold 0 ep 11 | loss 1.108 | S 0.7864 (6-class AUC 0.7862) | best 0.7932 | 10s
fold 0 ep 12 | loss 0.884 | S 0.7948 (6-class AUC 0.7920) | best 0

In [ ]:
# Idea: matched evaluation of InceptionTime under the BEATs protocol
# Goal: 5-fold ensemble on the 1,956 test recordings + OOF Accuracy/F1-optimal thresholds
# Source: threshold moving on OOF (Lipton et al. 2014)
oof = np.zeros(Ytr.shape, dtype=np.float32)
for k in range(5):
    oof[fold == k] = np.load(f'{OUT}/fold{k}_val_prob.npy')

probs = []
for k in range(5):
    m = InceptionTime().to(dev); m.load_state_dict(torch.load(f'{OUT}/fold{k}_best.pt', map_location=dev))
    probs.append(predict(m, Xte)); print('fold', k, 'predicted'); del m
P = np.mean(probs, 0); np.save(f'{OUT}/test_prob_5fold.npy', P)

T = {o: np.array([best_t(Ytr[:, c], oof[:, c], o) for c in range(7)]) for o in ['acc', 'f1']}
allrows = []
for o in ['acc', 'f1']:
    rows = []
    for c in range(7):
        y, p = Yte[:, c], P[:, c]; yp = p >= T[o][c]
        tp = (yp & (y == 1)).sum(); fp = (yp & (y == 0)).sum(); fn = (~yp & (y == 1)).sum()
        pr = tp / max(tp + fp, 1); rc = tp / max(tp + fn, 1)
        rows.append(dict(Class=CLS[c], Threshold=T[o][c], Accuracy=(yp == y).mean(), Precision=pr,
                         Recall=rc, F1=2*pr*rc / max(pr + rc, 1e-9), AUC=roc_auc_score(y, p)))
    df = pd.DataFrame(rows); df.insert(0, 'Objective', o); allrows.append(df)
    print(f'\n=== {o.upper()}-optimal thresholds ==='); print(df.drop(columns='Objective').to_string(index=False))
    mac = df[df.Class != 'Abnormal'][['Accuracy','Precision','Recall','F1','AUC']].mean().add_prefix('Macro-')
    print(mac.to_frame().T.to_string(index=False))
pd.concat(allrows).to_csv(f'{OUT}/inceptiontime_test_metrics.csv', index=False)
print('\nOOF 6-class Macro-AUC:', round(np.mean([roc_auc_score(Ytr[:, c], oof[:, c]) for c in range(6)]), 4))

fold 0 predicted
fold 1 predicted
fold 2 predicted
fold 3 predicted
fold 4 predicted

=== ACC-optimal thresholds ===
     Class  Threshold  Accuracy  Precision  Recall    F1   AUC
Inhalation       0.34     0.990      0.990   1.000 0.995 0.902
Exhalation       0.68     0.702      0.697   0.684 0.691 0.773
   Crackle       0.75     0.739      0.285   0.258 0.271 0.634
    Wheeze       0.73     0.792      0.491   0.141 0.219 0.604
   Stridor       0.99     0.994      0.000   0.000 0.000 0.787
   Rhonchi       0.88     0.852      0.640   0.054 0.100 0.578
  Abnormal       0.62     0.700      0.688   0.707 0.698 0.765
 Macro-Accuracy  Macro-Precision  Macro-Recall  Macro-F1  Macro-AUC
          0.845            0.517         0.356     0.379      0.713

=== F1-optimal thresholds ===
     Class  Threshold  Accuracy  Precision  Recall    F1   AUC
Inhalation       0.34     0.990      0.990   1.000 0.995 0.902
Exhalation       0.66     0.702      0.687   0.709 0.698 0.773
   Crackle       0.59  

In [ ]:
# Idea: unified matched comparison of the five models (Table 5) from saved probabilities
# Goal: identical OOF thresholding and test metrics for BEATs, AST, EfficientAT, CNN-BiGRU, InceptionTime
# Source: threshold moving on OOF (Lipton et al. 2014)
D = '/content/drive/MyDrive'
src = {'BEATs':         (f'{D}/beats_5fold/beats_oof_prob.npy', f'{D}/beats_5fold/beats_test_prob.npy'),
       'AST':           (None, f'{D}/matched_baselines/ast/test_prob_5fold.npy'),
       'EfficientAT':   (None, f'{D}/matched_baselines/efficientat/test_prob_5fold.npy'),
       'CNN-BiGRU':     (None, f'{D}/matched_baselines/crnn/test_prob_5fold.npy'),
       'InceptionTime': (None, f'{D}/matched_baselines/inceptiontime/test_prob_5fold.npy')}
fdir = {'AST': 'ast', 'EfficientAT': 'efficientat', 'CNN-BiGRU': 'crnn', 'InceptionTime': 'inceptiontime'}

rows = []
for name, (fo, ft) in src.items():
    if fo is None:                                             # rebuild OOF from per-fold validation files
        oof = np.zeros(Ytr.shape, dtype=np.float32)
        for k in range(5): oof[fold == k] = np.load(f'{D}/matched_baselines/{fdir[name]}/fold{k}_val_prob.npy')
    else:
        oof = np.load(fo)
    P = np.load(ft)
    assert oof.shape == Ytr.shape and P.shape == Yte.shape, (name, oof.shape, P.shape)
    for o in ['acc', 'f1']:
        t = np.array([best_t(Ytr[:, c], oof[:, c], o) for c in range(7)])
        m = []
        for c in range(6):
            y, p = Yte[:, c], P[:, c]; yp = p >= t[c]
            tp = (yp & (y == 1)).sum(); fp = (yp & (y == 0)).sum(); fn = (~yp & (y == 1)).sum()
            pr = tp / max(tp + fp, 1); rc = tp / max(tp + fn, 1)
            m.append([(yp == y).mean(), pr, rc, 2*pr*rc / max(pr + rc, 1e-9), roc_auc_score(y, p)])
        rows.append([name, o] + list(np.mean(m, 0)) + [roc_auc_score(Yte[:, 6], P[:, 6])])
tab = pd.DataFrame(rows, columns=['Model','Threshold','Macro-Accuracy','Macro-Precision','Macro-Recall','Macro-F1','Macro-AUC','Abnormal AUC'])
tab.to_csv(f'{D}/matched_baselines/table5_matched.csv', index=False)
for o in ['acc', 'f1']:
    print(f'\n=== {o.upper()}-optimal thresholds (6 classes, test n=1,956) ===')
    print(tab[tab.Threshold == o].drop(columns='Threshold').to_string(index=False, float_format='%.3f'))


=== ACC-optimal thresholds (6 classes, test n=1,956) ===
        Model  Macro-Accuracy  Macro-Precision  Macro-Recall  Macro-F1  Macro-AUC  Abnormal AUC
        BEATs           0.877            0.642         0.508     0.522      0.828         0.866
          AST           0.859            0.596         0.457     0.469      0.805         0.860
  EfficientAT           0.851            0.595         0.445     0.459      0.763         0.822
    CNN-BiGRU           0.856            0.580         0.452     0.464      0.767         0.778
InceptionTime           0.845            0.517         0.356     0.379      0.713         0.765

=== F1-optimal thresholds (6 classes, test n=1,956) ===
        Model  Macro-Accuracy  Macro-Precision  Macro-Recall  Macro-F1  Macro-AUC  Abnormal AUC
        BEATs           0.855            0.584         0.599     0.541      0.828         0.866
          AST           0.813            0.555         0.603     0.506      0.805         0.860
  EfficientAT        

In [ ]:
# Idea: matched temporal comparison — BEATs vs. the matched CNN-BiGRU, same folds (0,1), same rules
# Goal: make Table 7 use the same CNN-BiGRU as Table 5 and the same ensemble size as BEATs
# Source: event rules: IoU>=0.5 (Hsu et al. 2021), 200-ms collar (Mesaros et al. 2016)
import pandas as pd
from sklearn.metrics import roc_auc_score
D, FE = '/content/drive/MyDrive', '/content/drive/MyDrive/beats_5fold/frame_eval'
CLS6 = ['Inhalation','Exhalation','Crackle','Wheeze','Stridor','Rhonchi']
grid = np.round(np.arange(0.01, 1.0, 0.01), 2)
def acc_t(y, p): return grid[int(np.argmax([((p >= t) == y).mean() for t in grid]))]
def smooth(P): return F.avg_pool1d(torch.from_numpy(P).permute(0, 2, 1), 9, 1, 4).permute(0, 2, 1).numpy()

# 1) frame probabilities: matched CNN-BiGRU folds 0,1 and BEATs folds 0,1 (mean, then 9-frame moving average)
probs = []
for k in (0, 1):
    m = CRNN().to(dev); m.load_state_dict(torch.load(f'{D}/matched_baselines/crnn/fold{k}_best.pt', map_location=dev)); m.eval(); out = []
    with torch.no_grad(), torch.autocast('cuda', dtype=torch.float16):
        for b in range(0, len(Xte), 64):
            out.append(torch.sigmoid(m(torch.from_numpy(Xte[b:b+64].astype(np.float32)).to(dev)).float()).cpu().numpy())
    probs.append(np.concatenate(out)); del m
Pm = {'BEATs': smooth(np.load(f'{FE}/beats_test_frame_prob_folds01.npy').astype(np.float32).mean(0)),
      'CNN-BiGRU': smooth(np.mean(probs, 0).astype(np.float32))}

# 2) Accuracy-optimal thresholds from each model's own recording-level OOF probabilities
oof_c = np.zeros(Ytr.shape, dtype=np.float32)
for k in range(5): oof_c[fold == k] = np.load(f'{D}/matched_baselines/crnn/fold{k}_val_prob.npy')
oof = {'BEATs': np.load(f'{D}/beats_5fold/beats_oof_prob.npy'), 'CNN-BiGRU': oof_c}
T = {n: [acc_t(Ytr[:, c], oof[n][:, c]) for c in range(6)] for n in Pm}

# 3) event extraction and one-to-one matching
def runs(v):
    dv = np.diff(np.r_[0, v.astype(np.int8), 0]); return list(zip(np.where(dv == 1)[0], np.where(dv == -1)[0]))
def match(G, Q, rule):
    cand = []
    for i, (gs, ge) in enumerate(G):
        for j, (ps, pe) in enumerate(Q):
            iou = max(0, min(ge, pe) - max(gs, ps)) / (max(ge, pe) - min(gs, ps))
            ok = iou >= 0.5 if rule == 'iou' else (abs(ps - gs) <= 12.5 and abs(pe - ge) <= max(12.5, 0.2 * (ge - gs)))
            if ok: cand.append((iou, i, j))
    used_g, used_p, pairs = set(), set(), []
    for _, i, j in sorted(cand, reverse=True):
        if i not in used_g and j not in used_p: used_g.add(i); used_p.add(j); pairs.append((i, j))
    return pairs

rows, timing = [], []
for n, P in Pm.items():
    on_err, off_err = [], []
    for c in range(6):
        y, p = Yf_te[..., c].ravel(), P[..., c].ravel(); yp = p >= T[n][c]
        tp = (yp & (y == 1)).sum(); fF1 = 2 * tp / max(yp.sum() + y.sum(), 1)
        cnt = {'iou': [0, 0, 0], 'collar': [0, 0, 0]}                     # matched, predicted, reference
        for r in range(len(Xte)):
            G, Q = runs(Yf_te[r, :, c]), runs(P[r, :, c] >= T[n][c])
            for rule in cnt:
                pr = match(G, Q, rule); cnt[rule][0] += len(pr); cnt[rule][1] += len(Q); cnt[rule][2] += len(G)
                if rule == 'iou':
                    on_err += [abs(Q[j][0] - G[i][0]) * 16 for i, j in pr]; off_err += [abs(Q[j][1] - G[i][1]) * 16 for i, j in pr]
        ev = {}
        for rule, (tpm, npred, nref) in cnt.items():
            pr_, rc_ = tpm / max(npred, 1), tpm / max(nref, 1); ev[rule] = (pr_, rc_, 2 * pr_ * rc_ / max(pr_ + rc_, 1e-9))
        rows.append(dict(Model=n, Class=CLS6[c], FrameAUC=roc_auc_score(y, p), FrameF1=fF1,
                         EvP_IoU=ev['iou'][0], EvR_IoU=ev['iou'][1], EvF1_IoU=ev['iou'][2],
                         EvP_collar=ev['collar'][0], EvR_collar=ev['collar'][1], EvF1_collar=ev['collar'][2]))
    timing.append(dict(Model=n, median_onset_ms=np.median(on_err), median_offset_ms=np.median(off_err), matched=len(on_err)))

df = pd.DataFrame(rows); df.to_csv(f'{FE}/frame_event_beats_vs_crnn_matched.csv', index=False)
pd.set_option('display.precision', 3); pd.set_option('display.width', 200)
print(df.to_string(index=False))
print('\nMacro:'); print(df.groupby('Model').mean(numeric_only=True).to_string())
print('\nTiming (IoU-matched events):'); print(pd.DataFrame(timing).to_string(index=False))

    Model      Class  FrameAUC  FrameF1  EvP_IoU  EvR_IoU  EvF1_IoU  EvP_collar  EvR_collar  EvF1_collar
    BEATs Inhalation     0.966    0.748    0.660    0.915     0.767       0.451       0.625        0.524
    BEATs Exhalation     0.875    0.496    0.353    0.482     0.407       0.199       0.272        0.230
    BEATs    Crackle     0.926    0.510    0.340    0.465     0.392       0.232       0.318        0.268
    BEATs     Wheeze     0.949    0.514    0.484    0.391     0.433       0.403       0.325        0.360
    BEATs    Stridor     0.908    0.000    0.000    0.000     0.000       0.000       0.000        0.000
    BEATs    Rhonchi     0.820    0.181    0.411    0.077     0.129       0.280       0.052        0.088
CNN-BiGRU Inhalation     0.956    0.591    0.483    0.621     0.544       0.083       0.107        0.093
CNN-BiGRU Exhalation     0.864    0.482    0.443    0.464     0.453       0.193       0.202        0.198
CNN-BiGRU    Crackle     0.894    0.446    0.306    0.2

In [ ]:
# Idea: inspect the structure of the reference event file before using it
# Goal: read annotated test events exactly as stored (no frame-based merging)
# Source: -
import json
ev = json.load(open(f'{FE}/test_events.json'))
print(type(ev), len(ev))
first = list(ev.items())[0] if isinstance(ev, dict) else ev[0]
print(str(first)[:600])
names = np.load(f'{C}/test_names.npy', allow_pickle=True) if os.path.exists(f'{C}/test_names.npy') else None
print('test_names:', None if names is None else (names.shape, names[:3]))

<class 'list'> 1956
[[0, 1.12, 1.937], [2, 1.12, 1.937], [0, 3.34, 4.157], [2, 3.34, 4.157], [0, 5.674, 6.491], [2, 5.674, 6.491], [0, 7.855, 8.805], [2, 7.855, 8.805], [0, 10.018, 10.968], [2, 10.018, 10.968], [0, 12.105, 13.055], [2, 12.105, 13.055]]
test_names: ((1956,), array(['steth_20190801_09_46_05', 'steth_20190801_09_46_26',
       'steth_20190801_09_46_51'], dtype='<U31'))


In [ ]:
# Idea: matched temporal comparison using the annotated reference events (test_events.json)
# Goal: same reference events, matching rules and folds (0,1) for BEATs and the matched CNN-BiGRU
# Source: IoU>=0.5 (Hsu et al. 2021); 200-ms collar (Mesaros et al. 2016)
REF = json.load(open(f'{FE}/test_events.json'))                       # per recording: [class, onset_s, offset_s]

def pred_events(v, conv):                                             # frame runs -> (onset_s, offset_s)
    dv = np.diff(np.r_[0, v.astype(np.int8), 0]); s, e = np.where(dv == 1)[0], np.where(dv == -1)[0]
    return [(a * .016, (b if conv == 'excl' else b - 1) * .016) for a, b in zip(s, e)]

def match_s(G, Q, rule):
    cand = []
    for i, (gs, ge) in enumerate(G):
        for j, (ps, pe) in enumerate(Q):
            u = max(ge, pe) - min(gs, ps); iou = max(0, min(ge, pe) - max(gs, ps)) / u if u > 0 else 0
            ok = iou >= 0.5 if rule == 'iou' else (abs(ps - gs) <= 0.2 and abs(pe - ge) <= max(0.2, 0.2 * (ge - gs)))
            if ok: cand.append((iou, i, j))
    ug, up, pairs = set(), set(), []
    for _, i, j in sorted(cand, reverse=True):
        if i not in ug and j not in up: ug.add(i); up.add(j); pairs.append((i, j))
    return pairs

def evaluate(n, conv):
    rows, on_e, off_e = [], [], []
    for c in range(6):
        cnt = {'iou': [0, 0, 0], 'collar': [0, 0, 0]}
        for r in range(len(REF)):
            G = [(a, b) for k, a, b in REF[r] if k == c]; Q = pred_events(Pm[n][r, :, c] >= T[n][c], conv)
            for rule in cnt:
                pr = match_s(G, Q, rule); cnt[rule][0] += len(pr); cnt[rule][1] += len(Q); cnt[rule][2] += len(G)
                if rule == 'iou':
                    on_e += [abs(Q[j][0] - G[i][0]) * 1000 for i, j in pr]; off_e += [abs(Q[j][1] - G[i][1]) * 1000 for i, j in pr]
        row = dict(Model=n, Class=CLS6[c])
        for rule, (tp, npd, nrf) in cnt.items():
            p_, r_ = tp / max(npd, 1), tp / max(nrf, 1)
            row.update({f'EvP_{rule}': p_, f'EvR_{rule}': r_, f'EvF1_{rule}': 2 * p_ * r_ / max(p_ + r_, 1e-9)})
        rows.append(row)
    return pd.DataFrame(rows), np.median(on_e), np.median(off_e), len(on_e)

# choose the frame-to-time convention that reproduces the published BEATs values (IoU F1 0.349, collar 0.244)
chk = {cv: evaluate('BEATs', cv) for cv in ['excl', 'incl']}
for cv, (d_, on, off, nm) in chk.items():
    print(cv, '| BEATs macro IoU F1 %.3f | collar F1 %.3f | onset %.0f ms | offset %.0f ms' % (d_.EvF1_iou.mean(), d_.EvF1_collar.mean(), on, off))
conv = min(chk, key=lambda cv: abs(chk[cv][0].EvF1_iou.mean() - 0.349) + abs(chk[cv][0].EvF1_collar.mean() - 0.244))
print('convention used:', conv)

dB, onB, offB, nB = chk[conv]; dC, onC, offC, nC = evaluate('CNN-BiGRU', conv)
frame = df[['Model', 'Class', 'FrameAUC', 'FrameF1']]
res = frame.merge(pd.concat([dB, dC]), on=['Model', 'Class'])
res.to_csv(f'{FE}/frame_event_beats_vs_crnn_matched.csv', index=False)
print(res.to_string(index=False))
print('\nMacro:'); print(res.groupby('Model').mean(numeric_only=True).to_string())
print('\nTiming (IoU-matched): BEATs onset %.0f / offset %.0f ms (n=%d) | CNN-BiGRU onset %.0f / offset %.0f ms (n=%d)' % (onB, offB, nB, onC, offC, nC))

excl | BEATs macro IoU F1 0.350 | collar F1 0.242 | onset 91 ms | offset 122 ms
incl | BEATs macro IoU F1 0.350 | collar F1 0.244 | onset 91 ms | offset 113 ms
convention used: incl
    Model      Class  FrameAUC  FrameF1  EvP_iou  EvR_iou  EvF1_iou  EvP_collar  EvR_collar  EvF1_collar
    BEATs Inhalation     0.966    0.748    0.657    0.911     0.763       0.454       0.629        0.527
    BEATs Exhalation     0.875    0.496    0.348    0.475     0.401       0.198       0.271        0.229
    BEATs    Crackle     0.926    0.510    0.338    0.452     0.387       0.231       0.310        0.265
    BEATs     Wheeze     0.949    0.514    0.472    0.379     0.420       0.402       0.323        0.358
    BEATs    Stridor     0.908    0.000    0.000    0.000     0.000       0.000       0.000        0.000
    BEATs    Rhonchi     0.820    0.181    0.411    0.075     0.127       0.269       0.049        0.083
CNN-BiGRU Inhalation     0.956    0.591    0.477    0.613     0.536       0.084    

In [ ]:
# Idea: recording-based error counts to replace the misread MLCM percentages (Section 4.3)
# Goal: per true class — positives, true-positive recall, and recordings receiving each false label (F1-optimal OOF thresholds)
# Source: MLCM (Heydarian et al., IEEE Access 2022)
from google.colab import drive; drive.mount('/content/drive')
import numpy as np, pandas as pd
D, C = '/content/drive/MyDrive', '/content/drive/MyDrive/crnn_cache'
CLS6 = ['Inhalation','Exhalation','Crackle','Wheeze','Stridor','Rhonchi']
Ytr = np.load(f'{C}/train_Yf.npy').max(1).astype(int); Yte = np.load(f'{C}/test_Yf.npy').max(1).astype(int)
oof, P = np.load(f'{D}/beats_5fold/beats_oof_prob.npy')[:, :6], np.load(f'{D}/beats_5fold/beats_test_prob.npy')[:, :6]

grid = np.round(np.arange(0.01, 1.0, 0.01), 2)
def f1_t(y, p):
    return grid[int(np.argmax([2*((p>=t)&(y==1)).sum() / max((p>=t).sum() + y.sum(), 1) for t in grid]))]
T = np.array([f1_t(Ytr[:, c], oof[:, c]) for c in range(6)]); print('F1-optimal thresholds:', T)
Yp = (P >= T).astype(int)

rows = []
for c in range(6):
    pos = Yte[:, c] == 1; n = pos.sum()
    row = {'True class': CLS6[c], 'Positive recordings': n, 'TP (recall)': f'{(Yp[pos, c]==1).sum()} ({(Yp[pos, c]==1).mean()*100:.0f}%)'}
    for j in range(6):
        if j == c: continue
        k = (pos & (Yp[:, j] == 1) & (Yte[:, j] == 0)).sum()          # c-positive recordings that received a false j label
        row[f'false {CLS6[j]}'] = f'{k} ({k/n*100:.0f}%)'
    row['no label predicted'] = int((pos & (Yp.sum(1) == 0)).sum())
    rows.append(row)
pd.set_option('display.width', 250)
print(pd.DataFrame(rows).to_string(index=False))

# MLCM row totals (contributions) for comparison with recording-based denominators
def mlcm_row_total(c):
    tot = 0
    for t, p in zip(Yte, Yp):
        if t[c] == 0: continue
        fp = (p == 1) & (t == 0); fn = (p == 0) & (t == 1)
        tot += (1 if p[c] == 1 else 0) + (fp.sum() if fn[c] else 0) + (1 if (fn[c] and fp.sum() == 0) else 0)
    return tot
print('\nApprox. MLCM row totals (contributions):', {CLS6[c]: mlcm_row_total(c) for c in (3, 4, 5)})

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
F1-optimal thresholds: [0.39 0.62 0.5  0.43 0.36 0.69]
True class  Positive recordings TP (recall) false Exhalation false Crackle false Wheeze false Stridor false Rhonchi  no label predicted false Inhalation
Inhalation                 1935 1935 (100%)        304 (16%)     524 (27%)     135 (7%)       67 (3%)       16 (1%)                   0              NaN
Exhalation                  950   753 (79%)              NaN     347 (37%)      78 (8%)       31 (3%)       11 (1%)                   0           5 (1%)
   Crackle                  368   297 (81%)         69 (19%)           NaN     36 (10%)        6 (2%)        8 (2%)                   0           0 (0%)
    Wheeze                  405   307 (76%)        100 (25%)      77 (19%)          NaN      51 (13%)        4 (1%)                   0          12 (3%)
   Stridor                   12      1 (8%)        

In [ ]:
# Idea: session-reconstruction and duplicate/overlap audit for the leakage-prevention claim (Section 3.1)
# Goal: document timestamp parsing, the 10-min rule, multichannel grouping, cross-split separation, and duplicate/overlapping audio
# Source: HF_Lung_V1 official split (Hsu et al., PLOS ONE 2021)
from google.colab import drive; drive.mount('/content/drive')
import numpy as np, re, hashlib, datetime as dt, collections
C = '/content/drive/MyDrive/crnn_cache'
ntr = np.load(f'{C}/train_names.npy', allow_pickle=True); nte = np.load(f'{C}/test_names.npy', allow_pickle=True)
fold = np.load(f'{C}/train_fold.npy')
print('examples:', [n for n in ntr if n.startswith('steth')][:2], [n for n in ntr if n.startswith('trunc')][:3])

# 1) timestamp parsing and session reconstruction
def parse(n):
    d = ''.join(re.findall(r'\d', re.sub(r'_L\d+$', '', n)))[:14]       # YYYYMMDDHHMMSS, channel suffix removed
    return dt.datetime.strptime(d, '%Y%m%d%H%M%S')
def sessions(names):
    sid, s = {}, 0
    tr = collections.defaultdict(list)
    for n in names:
        if n.startswith('trunc'): tr[parse(n)].append(n)
    for t, g in tr.items():
        for n in g: sid[n] = s
        s += 1
    st = sorted([n for n in names if n.startswith('steth')], key=parse); prev = None
    for n in st:
        t = parse(n)
        if prev is None or (t - prev) > dt.timedelta(minutes=10): s += 1
        sid[n] = s; prev = t
    return sid, tr
sid_tr, tr_groups = sessions(ntr); sid_te, _ = sessions(nte)
print('sessions: train %d | test %d' % (len(set(sid_tr.values())), len(set(sid_te.values()))))
print('trunc channels per session (train):', collections.Counter(len(g) for g in tr_groups.values()))
bad = [s for s in set(sid_tr.values()) if len(set(fold[[i for i, n in enumerate(ntr) if sid_tr[n] == s]])) > 1]
print('sessions split across folds:', len(bad))

# 2) cross-split separation
dtr, dte = {parse(n).date() for n in ntr}, {parse(n).date() for n in nte}
print('recording days: train %d | test %d | shared %d' % (len(dtr), len(dte), len(dtr & dte)))
both, _ = sessions(np.concatenate([ntr, nte])); mixed = collections.defaultdict(set)
for n in ntr: mixed[both[n]].add('train')
for n in nte: mixed[both[n]].add('test')
print('sessions containing both train and test files:', sum(len(v) == 2 for v in mixed.values()))

# 3) exact duplicates, near duplicates, and temporal overlap
Xtr = np.load(f'{C}/train_X.npy'); Xte = np.load(f'{C}/test_X.npy')
h = collections.Counter(hashlib.md5(x.tobytes()).hexdigest() for x in np.concatenate([Xtr, Xte]))
print('exact duplicate spectrograms (all 9,765):', sum(c - 1 for c in h.values() if c > 1))
emb = lambda X: np.concatenate([X.mean(2), X.std(2)], 1).astype(np.float32)
Etr, Ete = emb(Xtr), emb(Xte)
Etr /= np.linalg.norm(Etr, axis=1, keepdims=True); Ete /= np.linalg.norm(Ete, axis=1, keepdims=True)
nn_idx = (Ete @ Etr.T).argmax(1)
r = np.array([np.corrcoef(Xte[i].ravel().astype(np.float32), Xtr[j].ravel().astype(np.float32))[0, 1] for i, j in enumerate(nn_idx)])
print('nearest train neighbour per test recording — Pearson r: max %.3f | 99th pct %.3f | test recs with r > 0.95: %d | > 0.99: %d'
      % (r.max(), np.percentile(r, 99), (r > 0.95).sum(), (r > 0.99).sum()))
for names, tag in [(ntr, 'train'), (nte, 'test')]:
    st = sorted([parse(n) for n in names if n.startswith('steth')])
    print(f'{tag}: consecutive steth files starting < 15 s apart (possible audio overlap):', sum((b - a).total_seconds() < 15 for a, b in zip(st, st[1:])))

Mounted at /content/drive
examples: [np.str_('steth_20180814_09_37_11'), np.str_('steth_20180814_09_37_33')] [np.str_('trunc_2019-05-07-13-57-26-L1_0'), np.str_('trunc_2019-05-07-13-57-26-L1_1'), np.str_('trunc_2019-05-07-13-57-26-L1_2')]
sessions: train 382 | test 143
trunc channels per session (train): Counter({86: 4, 88: 4, 87: 4, 90: 3, 64: 3, 63: 3, 18: 2, 85: 2, 48: 2, 75: 2, 58: 2, 67: 2, 70: 2, 73: 2, 51: 2, 30: 1, 36: 1, 55: 1, 96: 1, 60: 1, 57: 1, 44: 1, 9: 1, 1: 1, 82: 1, 72: 1, 91: 1, 74: 1, 81: 1, 83: 1, 84: 1, 8: 1, 102: 1, 53: 1, 54: 1, 71: 1, 80: 1, 89: 1, 78: 1, 95: 1, 16: 1, 79: 1, 25: 1, 65: 1})
sessions split across folds: 0
recording days: train 118 | test 39 | shared 0
sessions containing both train and test files: 0
exact duplicate spectrograms (all 9,765): 0
nearest train neighbour per test recording — Pearson r: max 0.841 | 99th pct 0.813 | test recs with r > 0.95: 0 | > 0.99: 0
train: consecutive steth files starting < 15 s apart (possible audio overlap): 137


In [ ]:
# Idea: extend the rule search (analysis-window overlap) and report class-wise agreement for the best rule
# Goal: exact label-to-frame rule + class-wise, positive-frame, event-level and boundary agreement
# Source: HF_Lung_V1 (Hsu et al., PLOS ONE 2021)
EXTRA = {'window overlap ±32 ms': lambda on, off: (c_ + 0.032 > on) & (c_ - 0.032 < off),
         'window overlap ±64 ms (n_fft 512)': lambda on, off: (c_ + 0.064 > on) & (c_ - 0.064 < off),
         'ceil(on)..floor(off)': lambda on, off: (k >= np.ceil(on / .016)) & (k <= np.floor(off / .016))}
for r, f in EXTRA.items():
    ys = []
    for ev in events:
        y = np.zeros((938, 6), bool)
        for c, on, off in ev: y[:, c] |= f(on, off)
        ys.append(y)
    ref[r] = ys; RULES[r] = f
    R = np.stack(ys); print(f'{r:34s} overall agreement {100 * (R == cache).mean():.4f}% | differing frames {(R != cache).sum()}')

best = max(RULES, key=lambda r: (np.stack(ref[r]) == cache).mean()); R = np.stack(ref[best]); print('\nbest rule:', best)
rows = []
for c in range(6):
    a, b = R[..., c], cache[..., c]; diff = a != b
    bnd = np.zeros_like(diff)                                             # frames within one frame of an event boundary
    for i, ev in enumerate(events):
        for cc, on, off in ev:
            if cc == c:
                for t in (on, off): j = int(round(t / .016)); bnd[i, max(0, j - 1):j + 2] = True
    evs = [(i, on, off) for i, ev in enumerate(events) for cc, on, off in ev if cc == c]
    hit = sum(b[i, int(np.floor(on / .016)):int(np.ceil(off / .016)) + 1].any() for i, on, off in evs)
    rows.append({'Class': CLS6[c], 'Overall %': 100 * (~diff).mean(), 'Differing frames': int(diff.sum()),
                 'at boundaries': int((diff & bnd).sum()), 'only in cache': int((b & ~a).sum()), 'only in rebuild': int((a & ~b).sum()),
                 'Positive-frame agreement (Jaccard) %': 100 * (a & b).sum() / max((a | b).sum(), 1),
                 'Events': len(evs), 'Events with >=1 cached positive frame %': 100 * hit / max(len(evs), 1)})
pd.set_option('display.width', 300); print(pd.DataFrame(rows).to_string(index=False, float_format='%.3f'))

window overlap ±32 ms              overall agreement 99.5762% | differing frames 232915
window overlap ±64 ms (n_fft 512)  overall agreement 99.0233% | differing frames 536769
ceil(on)..floor(off)               overall agreement 99.8542% | differing frames 80120

best rule: floor(on)..ceil(off)
     Class  Overall %  Differing frames  at boundaries  only in cache  only in rebuild  Positive-frame agreement (Jaccard) %  Events  Events with >=1 cached positive frame %
Inhalation     99.638             33150          33150              0            33150                                98.379   34095                                  100.000
Exhalation     99.808             17612          17612              0            17612                                98.445   18349                                  100.000
   Crackle     99.847             13971          13971              0            13971                                98.437   15606                                  100.000
    Whee

In [ ]:
# Idea: confirm the half-open rule floor(on/16 ms) <= k < ceil(off/16 ms)
# Goal: exact reproduction of the cached frame labels from the original annotations
# Source: -
R = np.stack([np.stack([np.any([(k >= np.floor(on / .016)) & (k < np.ceil(off / .016)) for cc, on, off in ev if cc == c] or [np.zeros(938, bool)], 0)
                        for c in range(6)], 1) for ev in events])
d = R != cache
print('half-open rule — differing frames: %d | overall agreement %.5f%%' % (d.sum(), 100 * (~d).mean()))
print('per class differing frames:', dict(zip(CLS6, d.sum((0, 1)).tolist())))
print('positive frames per class (cache):', dict(zip(CLS6, cache.sum((0, 1)).tolist())))

half-open rule — differing frames: 0 | overall agreement 100.00000%
per class differing frames: {'Inhalation': 0, 'Exhalation': 0, 'Crackle': 0, 'Wheeze': 0, 'Stridor': 0, 'Rhonchi': 0}
positive frames per class (cache): {'Inhalation': 2011368, 'Exhalation': 1114758, 'Crackle': 879815, 'Wheeze': 455074, 'Stridor': 36106, 'Rhonchi': 236831}
